<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

## SEC Executive Extraction Pipeline

## Project Overview
**Task**: Extract executive information (names and titles) from SEC filings  
**Dataset**: 8k_filings_raw_text_2024  
**Goal**: Build a proprietary database of company executives from official SEC filings

---

## What This Pipeline Does?
1. **Test Mode**: Analyzes first 10 rows to validate extraction patterns
2. **Full Processing**: Extracts executives from entire dataset in chunks
3. **Cleaning**: Clean the raw that with the level of confidence (low, spacy)
4. **Normalize**: Normalize all text (if possible)

---

## Libraries Used

- **Packages**: `pandas`, `psutil`, `tqdm`, `spaCy`, `re`, `csv`, `os`, `time`, `pathlib`, `datetime`, `typing`, `collections`

---


## Pipeline

### Step 1: Configure / Knowing the data
```
INPUT_FILE = "8k_filings_raw_text_2024.csv"
FINAL_FILE = NONE
```
### Step 2: Extraction Process
```
INPUT_FILE = "8k_filings_raw_text_2024.csv"
FINAL_FILE = "executive_raw.csv"
```
### Step 3: Depuration
- Stage 1: Low confidence depuration
```
INPUT_FILE = "executive_raw.csv"
FINAL_FILE = "executive_cleaned_low.csv"
```
- Stage 2: spaCy confidence depuration
```
INPUT_FILE = "executive_cleaned_low.csv"
FINAL_FILE = "executive_cleaned_spacy.csv"
```
### Step 3: Noramlization
```
INPUT_FILE = "executive_cleaned_spacy.csv"
FINAL_FILE = "executive_final.csv"
```

# SET-UP / KNOWING THE DATA

In [2]:
"""
SEC Executive Extraction Pipeline
Author: David Acevedo-Cardona
"""

#Imports:
import pandas as pd
import re, csv, spacy, os, time
from pathlib import Path
from datetime import datetime
from typing import List, Tuple
from collections import Counter
from tqdm.notebook import tqdm

In [3]:
#Print some values:

filings_df = "8k_filings_raw_text_2024.csv"

# Display some rows
df_sample = pd.read_csv(filings_df, nrows=10)

#Print
df_sample

,sec_accession_number,release_datetime,title,sec_filing_type,keywords,exchange,symbol,company_name,excerpt,raw_text
0,0000950170-24-000282,2024-01-03 08:29:17+11:00,Tesla Reports Record Vehicle Production and De...,8-K,"Tesla,Vehicle Production,Vehicle Deliveries,Fi...",NASDAQ,TSLA,"Tesla, Inc.",Tesla announced record vehicle production and ...,=== MAIN 8-K FILING ===\n8-K false-01-022024-0...
1,0001193125-24-005943,2024-01-11 08:16:32+11:00,Berkshire Hathaway Settles Delaware Litigation...,8-K,"litigation,settlement,Berkshire Hathaway,Pilot...",NYSE,BRK-B,BERKSHIRE HATHAWAY INC,Berkshire Hathaway has reached a full settleme...,=== MAIN 8-K FILING ===\n8-K BERKSHIRE HATHAWA...
2,0001213900-24-002759,2024-01-11 09:44:11+11:00,Steel Partners Holdings L.P. Abandons Previous...,8-K,"unit split,reverse split,forward split,share r...",NYSE,SPLP,STEEL PARTNERS HOLDINGS L.P.,Steel Partners Holdings L.P. has announced the...,=== MAIN 8-K FILING ===\nfalse 2024-01-10 20...
3,0001213900-24-002761,2024-01-11 09:45:04+11:00,Akerna Sets Special Meeting Date for Gryphon D...,8-K,"Merger,Akerna,Gryphon Digital Mining,Stockhold...",NASDAQ,GRYP,"Gryphon Digital Mining, Inc.",Akerna has announced the date for a special st...,=== MAIN 8-K FILING ===\nfalse 2024-01-10 20...
4,0001683168-24-000184,2024-01-11 09:47:10+11:00,"Focus Universal Secures $300,000 Loan as Part ...",8-K,"revolving credit facility,loan,financing,debt,...",NASDAQ,FCUV,FOCUS UNIVERSAL INC.,"Focus Universal Inc. has entered into a $300,0...",=== MAIN 8-K FILING ===\nfalse 2024-01-09 20...
5,0001021771-24-000018,2024-01-11 09:53:18+11:00,Kingstone Companies Issues Letter to Stockhold...,8-K,"Kingstone Companies,stockholders,strategic rev...",NASDAQ,KINS,"KINGSTONE COMPANIES, INC.","Kingstone Companies, Inc. announced the issuan...",=== MAIN 8-K FILING ===\nfalse-01-102024-01-10...
6,0001564708-24-000028,2024-01-11 10:43:55+11:00,News Corp Continues $1 Billion Share Repurchas...,8-K,"share repurchase,stock buyback,News Corporatio...",NASDAQ,NWSA,NEWS CORP,News Corporation has provided an update on its...,=== MAIN 8-K FILING ===\nnws-2024011false-01-1...
7,0001609711-24-000006,2024-01-11 11:51:48+11:00,GoDaddy Announces $1.752 Billion Refinancing o...,8-K,"Refinancing,Term Loans,Debt,Credit Agreement,G...",NYSE,GDDY,GoDaddy Inc.,GoDaddy has announced a $1.752 billion refinan...,=== MAIN 8-K FILING ===\ngddy-2024011false-01-...
8,0001193125-24-006124,2024-01-11 12:18:31+11:00,Hilton Grand Vacations Prices $900 Million Sen...,8-K,"Hilton Grand Vacations,Senior Secured Notes,Bl...",NYSE,HGV,Hilton Grand Vacations Inc.,Hilton Grand Vacations has priced a $900 milli...,=== MAIN 8-K FILING ===\n8-K false 2024-01-1...
9,0001104659-24-003089,2024-01-11 12:36:12+11:00,EchoStar Unlocks Strategic Flexibility Post-Me...,8-K,"EchoStar,DISH Network,Merger,Wireless Spectrum...",NASDAQ,SATS,EchoStar CORP,EchoStar Corporation has completed strategic t...,=== MAIN 8-K FILING ===\nfalse false 8-K 202...


In [4]:
class ExecutiveExtractor:
    """
    Extract executive information (names and titles) from SEC filing text.
    Combines regex and NLP (spaCy) methods for robust extraction.
    """

    def __init__(self):
        """Initialize spaCy model and executive title list."""
        self.nlp = spacy.load("en_core_web_sm")
        self.nlp.max_length = 4_000_000  # allow up to ~4M characters per chunk

        self.exec_titles = [
            'CEO', 'CFO', 'COO', 'CTO', 'CLO',
            'Chief Executive Officer', 'Chief Financial Officer',
            'Chief Operating Officer', 'Chief Technology Officer',
            'President', 'Vice President', 'Senior Vice President',
            'General Counsel', 'Corporate Secretary',
            'Chairman', 'Director', 'Treasurer'
        ]

    # MAIN EXTRACTION FUNCTION
    def extract_executives(self, text: str) -> List[Tuple[str, str, str]]:
        """Run regex + spaCy extraction and merge results."""
        regex_ans = self.regex_extract(text)
        spacy_ans = self.spacy_extract(text)
        merged = regex_ans + spacy_ans
        return self.deduplicate_executives(merged)

    # REGEX EXTRACTION
    def regex_extract(self, text: str) -> List[Tuple[str, str, str]]:
        """Extract structured executive info via regex patterns."""
        executives = []

        # Pattern 1: multiline “By /s/ … Name … Title …”
        regex1 = re.compile(
            r'By:\s*/s/\s*([A-Z][a-zA-Z.\s]+?)\s*\n\s*'
            r'(?:Name:\s*)?([A-Z][a-zA-Z.\s]+?)\s*\n\s*'
            r'(?:Title:\s*)?([^\n]+?)(?=\n\s*Date:|\n\n|\Z)',
            re.MULTILINE
        )
        for match in regex1.finditer(text):
            name = match.group(2).strip()
            title = re.sub(r'\s*Date:.*', '', match.group(3)).strip()
            if len(name.split()) >= 2 and self.executive_title(title):
                executives.append((name, title, 'high'))

        # Pattern 2: “By: /s/ [Name]” + title nearby
        regex2 = re.compile(r'By:\s*/s/\s*([A-Z][a-zA-Z.\s]+?)', re.MULTILINE)
        for match in regex2.finditer(text):
            name = match.group(1).strip()
            next_text = text[match.end():match.end() + 200]
            title_match = re.search(
                r'([A-Z][a-zA-Z\s,&]+(?:Officer|President|Director|Counsel|Secretary|Chairman|Treasurer))',
                next_text
            )
            if title_match and len(name.split()) >= 2:
                title = title_match.group(1).strip()
                executives.append((name, title, 'medium'))

        # Pattern 3: inline “Name, Chief Executive Officer”
        regex3 = re.compile(
            r'([A-Z][a-z]+(?:\s+[A-Z]\.?)?\s+[A-Z][a-z]+),\s*'
            r'(Chief\s+\w+\s+Officer|President|Vice\s+President|General\s+Counsel|Director)',
            re.IGNORECASE
        )
        for match in regex3.finditer(text):
            name = match.group(1).strip()
            title = match.group(2).strip()
            if len(name.split()) >= 2 and self.executive_title(title):
                executives.append((name, title, 'low'))

        # Pattern 4: inline “By /s/ … Name: … Title: …”
        regex4 = re.compile(
            r'By:\s*/s/\s*([A-Z][a-zA-Z.\s]+?)\s*Name:\s*([A-Z][a-zA-Z.\s]+?)\s*Title:\s*([A-Za-z\s,&]+)',
            re.MULTILINE
        )
        for match in regex4.finditer(text):
            by_name = match.group(1).strip()
            true_name = match.group(2).strip()
            title = match.group(3).strip()
            name = true_name if len(true_name.split()) >= 2 else by_name
            if len(name.split()) >= 2 and self.executive_title(title):
                executives.append((name, title, 'high'))

        return executives

    # SPACY EXTRACTION
    def spacy_extract(self, text: str) -> List[Tuple[str, str, str]]:
        """Use spaCy to find PERSON entities near executive titles."""
        text = text.replace("\xa0", " ")  # normalize non-breaking spaces
        executives = []

        if len(text) > self.nlp.max_length:
            print(f"⚠️ Splitting long text ({len(text):,} chars) into chunks...")
            step = self.nlp.max_length - 50_000  # 50k overlap for safety
            parts = [text[i:i + step] for i in range(0, len(text), step)]
            for i, part in enumerate(parts, 1):
                print(f"  → Processing chunk {i}/{len(parts)} ({len(part):,} chars)")
                doc_part = self.nlp(part)
                executives.extend(self._extract_from_doc(doc_part, part))
            return executives

        doc = self.nlp(text)
        return self._extract_from_doc(doc, text)

    # HELPERS
    def _extract_from_doc(self, doc, text):
        executives = []
        exec_keywords = [
            'chief', 'officer', 'president', 'counsel',
            'secretary', 'chairman', 'treasurer', 'director',
            'ceo', 'cfo', 'coo', 'cto'
        ]
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                name_text = ent.text.strip()
                if len(name_text.split()) < 2:
                    continue
                # skip junk terms
                if any(w in name_text.lower() for w in ['item', 'section', 'exhibit', 'entry']):
                    continue
                context = text[max(0, ent.start_char - 150): ent.end_char + 150].lower()
                if any(kw in context for kw in exec_keywords):
                    title_match = re.search(
                        r'(ceo|chief\s+[a-z]+\s+officer|cfo|coo|cto|president|vice\s+president|'
                        r'general\s+counsel|corporate\s+secretary|chairman|director|treasurer)',
                        context, re.IGNORECASE
                    )
                    title = title_match.group(1).title() if title_match else "Unknown"
                    executives.append((name_text, title.strip(), 'spacy'))
        return executives

    def executive_title(self, title: str) -> bool:
        """Check if string contains executive role words."""
        keywords = [
            'chief', 'officer', 'president', 'counsel',
            'secretary', 'chairman', 'director', 'treasurer'
        ]
        return any(k in title.lower() for k in keywords)

    def deduplicate_executives(self, executives: List[Tuple[str, str, str]]):
        """Keep only highest-confidence entries per name."""
        if not executives:
            return []
        confidence_order = {'high': 0, 'medium': 1, 'spacy': 2, 'low': 3}
        name_dict = {}
        for name, title, conf in executives:
            key = name.lower().strip()
            if key not in name_dict or confidence_order[conf] < confidence_order[name_dict[key][2]]:
                name_dict[key] = (name, title, conf)
        return list(name_dict.values())

In [5]:
# Create extractor instance
extractor = ExecutiveExtractor()

# Test on a few filings
for i, row in df_sample.iterrows():
    text = str(row['raw_text'])
    results = extractor.extract_executives(text)
    if results:  # only show filings where something is found
        print(f"\n=== {row['company_name']} ({row['sec_filing_type']}) ===")
        for name, title, confidence in results:
            print(f"  ✓ {name} — {title} ({confidence})")


=== Tesla, Inc. (8-K) ===
  ✓ Brandon Ehrhart — General Counsel (spacy)

=== BERKSHIRE HATHAWAY INC (8-K) ===
  ✓ Marc D. Hamburg — Vice President (spacy)
  ✓ Berkshire Hathaway Reaches Settlement — Chief Financial Officer (spacy)

=== STEEL PARTNERS HOLDINGS L.P. (8-K) ===
  ✓ Ryan O’Herrin Ryan — Chief Financial Officer (spacy)

=== Gryphon Digital Mining, Inc. (8-K) ===
  ✓ Jessica Billingsley — Chief Executive Officer (high)
  ✓ Jessica Billingsley Name — Chief Executive Officer (spacy)
  ✓ Jessica Billingsley Title — Chief Executive Officer (spacy)
  ✓ Rob Chang — Ceo (spacy)

=== FOCUS UNIVERSAL INC. (8-K) ===
  ✓ Desheng Wang — Chief Executive Officer (high)
  ✓ Desheng Wang Name — Chief Executive Officer (spacy)
  ✓ Desheng Wang Title — Chief Executive Officer (spacy)

=== KINGSTONE COMPANIES, INC. (8-K) ===
  ✓ Jennifer Gravelle — Cfo (spacy)
  ✓ Gravelle CFO — Cfo (spacy)

=== NEWS CORP (8-K) ===
  ✓ Michael L. Bunder — Vice President (spacy)

=== GoDaddy Inc. (8-K) ===
  ✓ 

# FIRST EXTRACTION PROCESS

In [5]:
# Configuration
INPUT_PATH = "8k_filings_raw_text_2024.csv"
OUTPUT_PATH = "executive_raw.csv"

# Adjusted for 16 GB RAM
CHUNK_SIZE = 1_500_000           # process text in 1.5M-character pieces
extractor.nlp.max_length = 2_000_000  # allow max 2M characters per spaCy call

# Load input data
df = pd.read_csv(INPUT_PATH)

print(f"✅ Found {len(df)} filings to process.")
if os.path.exists(OUTPUT_PATH):
    processed = pd.read_csv(OUTPUT_PATH)
    done = set(processed["company"].unique())
    print(f"✅ Found {len(done)} companies already processed.")
else:
    done = set()
    print("🆕 No previous output found. Starting fresh.")

# ==============================
# Main Extraction Loop
# ==============================
results = []
completed = set(done)
new_executives = 0
chunk_counter = 0

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing SEC filings"):
    company = str(row["company_name"])
    filing = str(row["sec_filing_type"])
    text = str(row["raw_text"])

    # Skip previously processed companies
    if company in completed:
        continue

    # Split long texts safely
    if len(text) > extractor.nlp.max_length:
        print(f"\n⚠️ Splitting long text ({len(text):,} chars) into chunks...")
        parts = [text[i:i+CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]
    else:
        parts = [text]

    for part_id, part in enumerate(parts, 1):
        start_time = time.time()
        print(f"⏳ Processing chunk {part_id}/{len(parts)} for {company}...")

        try:
            execs = extractor.extract_executives(part)
        except Exception as e:
            print(f"❌ Error processing {company}, chunk {part_id}: {e}")
            continue

        elapsed = time.time() - start_time
        print(f"✅ Finished chunk {part_id}/{len(parts)} in {elapsed/60:.2f} min")

        for name, title, confidence in execs:
            results.append({
                "company": company,
                "filing_type": filing,
                "executive_name": name,
                "executive_title": title,
                "confidence": confidence
            })
        new_executives += len(execs)

    completed.add(company)
    chunk_counter += 1

    # Save progress every 100 companies
    if chunk_counter % 100 == 0 and results:
        df_out = pd.DataFrame(results)
        df_out.to_csv(OUTPUT_PATH, mode="a", index=False, header=not os.path.exists(OUTPUT_PATH))
        results.clear()
        print(f"💾 Progress saved after {chunk_counter} companies ({len(completed)} total).")

print("\n✅ Extraction complete!")
print(f"💾 Results saved to: {OUTPUT_PATH}")
print(f"🏢 Total unique companies processed: {len(completed)}")
print(f"👔 Total executives extracted: {new_executives}")

✅ Found 58126 filings to process.
✅ Found 5524 companies already processed.


Processing SEC filings:   0%|          | 0/58126 [00:00<?, ?it/s]

⏳ Processing chunk 1/1 for Transocean Ltd....
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for Lucid Group, Inc....
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for IDEAYA Biosciences, Inc....
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for HIMALAYA TECHNOLOGIES, INC...
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for MILESTONE SCIENTIFIC INC....
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for DatChat, Inc....
✅ Finished chunk 1/1 in 0.01 min
⏳ Processing chunk 1/1 for Forestar Group Inc....
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for CRISPR Therapeutics AG...
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for BAR HARBOR BANKSHARES...
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for Optimus Healthcare Services, Inc....
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for MIDDLESEX WATER CO...
✅ Finished chunk 1/1 in 0.00 min
⏳ Processing chunk 1/1 for MetroCity Bankshares, Inc....
✅ Fini

# DEPURATION

## Stage 1 = Low Confidence

In [17]:
# SET-UP
INPUT_PATH = "executive_raw.csv"    
OUTPUT_PATH = "executive_low.csv"

# Data
df = pd.read_csv(INPUT_PATH)

#HELPERS
role_terms = set("""
officer officers president vice-president vicepresident vice president chair chairman chairperson
director directors partner partners trustee trustees member members stockholder stockholders shareholder shareholders
advisor advisers adviser consultants consultant manager managers employee employees employer
counsel secretary treasurer controller comptroller attorney
affiliates affiliate subsidiary subsidiaries principal principals owner owners
board committee corporation corp inc llc ltd plc co company companies association
executive operating financial legal accounting compliance general corporate administrative
svp evp avp vp cfo ceo coo cto clo cso cio cro cmo cao gc
""".split())

particles = {"da","de","del","de la","di","du","la","le","van","von","der","den","dos","das","mac","mc","bin","ibn","al"}
suffixes = {"jr","sr","ii","iii","iv","v"}

def token_is_name_like(tok):
    tok = tok.replace("’", "'")
    plain = re.sub(r"[^A-Za-z\-']", "", tok)
    if not plain:
        return False
    low = plain.lower()
    if re.fullmatch(r"[A-Z]\.?", tok): return True
    if re.fullmatch(r"[A-Z][a-z]+(?:[-'][A-Z][a-z]+)+", tok): return True
    if re.fullmatch(r"[A-Z][a-z]+", tok) and low not in role_terms: return True
    if low in particles or low.strip(".") in suffixes: return True
    return False

name_pattern = re.compile(
    r"(?:[A-Z][a-z]+|[A-Z]\.?|[A-Z][a-z]+[-'][A-Z][a-z]+|O['’][A-Z][a-z]+)"
    r"(?:\s+(?:[A-Z][a-z]+|[A-Z]\.?|[A-Z][a-z]+[-'][A-Z][a-z]+|O['’][A-Z][a-z]+|"
    r"(?:da|de|del|de la|di|du|la|le|van|von|der|den|dos|das|mac|mc|bin|ibn|al)))+"
)

def extract_best_name(text):
    if pd.isna(text): return None
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text)).replace("’", "'")
    cands = []
    for m in name_pattern.finditer(text):
        span = m.group(0).strip()
        toks = span.split()
        if sum(token_is_name_like(t) for t in toks) >= max(2, len(toks)-1):
            if any(t.lower().strip(".") in role_terms for t in toks): continue
            cands.append(span)
    if not cands:
        return None
    cands.sort(key=lambda s: (abs(len(s.split())-2), len(s)))
    return cands[0]

#Process data
mask_low = df["confidence"].str.lower().eq("low")
df_low = df.loc[mask_low].copy()
df_low["executive_name"] = df_low["executive_name"].apply(extract_best_name)
df_low = df_low[df_low["executive_name"].notna()]

# Keep all the pther values
df_keep = df.loc[~mask_low].copy()
df_final = pd.concat([df_keep, df_low], ignore_index=True)

# Name filter
def looks_like_name(n):
    if pd.isna(n): return False
    n = str(n).strip()
    toks = n.split()
    if len(toks) < 2: return False
    if not all(token_is_name_like(t) for t in toks): return False
    return True

df_final = df_final[df_final["executive_name"].apply(looks_like_name)]

# --- Save ---
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved as {OUTPUT_PATH}")
print(f"Input: {len(df):,} rows → Output: {len(df_final):,} rows")
print("\nPreview:")
print(df_final.sample(10))


✅ Cleaned dataset saved as executive_low.csv
Input: 32,369 rows → Output: 22,645 rows

Preview:
                                             company filing_type  \
1838               African Agriculture Holdings Inc.         8-K   
28345                    ASPAC III Acquisition Corp.         8-K   
25912                  Black Spade Acquisition II Co         8-K   
26033                  Andretti Acquisition Corp. II         8-K   
14726                            AETHLON MEDICAL INC         8-K   
4912                     JACK HENRY & ASSOCIATES INC         8-K   
17063                              Gannett Co., Inc.         8-K   
27355  WESTERN ASSET EMERGING MARKETS DEBT FUND INC.         8-K   
19247                        Bath & Body Works, Inc.         8-K   
16128                        Cushman & Wakefield plc         8-K   

                 executive_name          executive_title confidence  
1838           Harry Green Name  Chief Financial Officer      spacy  
28345       Con

## Stage 2: spaCy

In [8]:
# Load SpaCy English model
nlp = spacy.load("en_core_web_sm")

# Load cleaned low data
raw_path = "executive_cleaned_low.csv"
df = pd.read_csv(raw_path)

# Extraction
def extract_persons(text):
    if pd.isna(text):
        return None
    doc = nlp(text)
    names = [ent.text.strip() for ent in doc.ents if ent.label_ == "PERSON"]
    return names[0] if names else None

df["executive_name"] = df["executive_name"].apply(extract_persons)

# --- STEP 3: Basic text cleaning ---
df["executive_name"] = (
    df["executive_name"]
    .astype(str)
    .str.replace(r"\b(Email|Phone)\b", "", regex=True)
    .str.strip()
)

# Remove false positives
false_positive_keywords = [
    "Prepared Remarks", "Good Standing", "Qualified Transferee", "Diligence", "Milestones",
    "Retirement", "Witness", "Indemnifying", "Securities", "Transfer Restricted", "Baton Rouge",
    "Ganado Advocates", "Due Diligence", "Mutual Acknowledgment", "Hasche Sigle",
    "Dykema Gossett", "Gunderson Dettmer", "Advocates", "Consulting", "LLP", "LLC", "Group",
    "Corp", "Holdings", "Capital", "Incorporated", "Partners", "PLC", "Advisors", "Associates",
    "Inc", "Company", "Enterprises", "Legal", "Strategy", "Bank", "Investments", "Management",
    "Services"
]
pattern = re.compile("|".join([re.escape(k) for k in false_positive_keywords]), re.IGNORECASE)

def is_valid_name(name):
    if pd.isna(name) or name.strip() == "":
        return False
    if pattern.search(name):
        return False
    if len(name.split()) > 6 or any(char.isdigit() for char in name):
        return False
    if name.isupper() and len(name) > 2:
        return False
    return True

df = df[df["executive_name"].apply(is_valid_name)]

# Delete duplicates
df = df.drop_duplicates(subset=["executive_name"]).reset_index(drop=True)

# --- STEP 6: Save final cleaned file ---
final_path = "executive_cleaned_spacy.csv"
df.to_csv(final_path, index=False)

print(f"Cleaned dataset saved as {final_path} ({len(df)} rows)")


✅ Cleaned dataset saved as executive_cleaned_spacy.csv (17626 rows)


# Normalize Text

In [11]:
# Load the file
df = pd.read_csv("executive_cleaned_spacy.csv")

# Function to normalize executive titles
def clean_title(title):
    if pd.isna(title):
        return None

    # Remove HTML tags, symbols, extra whitespace
    title = re.sub(r"<.*?>", " ", str(title))
    title = re.sub(r"[^A-Za-z\s&\-./]", " ", title)  # Keep letters and a few safe symbols
    title = re.sub(r"\s+", " ", title).strip()

    # Lowercase for normalization
    lower = title.lower()

    # Common junk or filler terms
    if lower in ["na", "n a", "none", "null", "—", "-", "", "0"]:
        return None

    # Normalize common abbreviations
    replacements = {
        r"\bceo\b|chief exec(utive)? officer": "Chief Executive Officer",
        r"\bcoo\b|chief operating officer": "Chief Operating Officer",
        r"\bcfo\b|chief financial officer": "Chief Financial Officer",
        r"\bcio\b|chief information officer": "Chief Information Officer",
        r"\bcto\b|chief technology officer": "Chief Technology Officer",
        r"\bchro\b|chief human resources officer": "Chief Human Resources Officer",
        r"\bcmo\b|chief marketing officer": "Chief Marketing Officer",
        r"\bcso\b|chief strategy officer": "Chief Strategy Officer",
        r"\bcro\b|chief risk officer": "Chief Risk Officer",
        r"\bvp\b|vice pres(ident)?": "Vice President",
        r"\bpresident\b": "President",
        r"\bchair(man|woman)?\b": "Chair",
        r"\bmanaging dir(ector)?\b": "Managing Director",
        r"\bexec(utive)? dir(ector)?\b": "Executive Director",
        r"\bboard member\b|director\b": "Director",
    }

    for pattern, replacement in replacements.items():
        if re.search(pattern, lower):
            return replacement

    # If not matched, Title Case the cleaned string
    return title.title()

# Apply cleaning
df["executive_title_clean"] = df["executive_title"].apply(clean_title)

# Drop duplicates if needed
df = df.drop_duplicates(subset=["executive_name", "executive_title_clean"]).reset_index(drop=True)

# Save normalized version
df.to_csv("executive_final.csv", index=False)

print("Text normalized saved as executive_final.csv")

Text normalized saved as executive_final.csv


# FINAL OVERVIEW

In [12]:
# Load final file
df = pd.read_csv("executive_final.csv")

# Identify key columns
name_col = [c for c in df.columns if "name" in c.lower()][0]
title_col = [c for c in df.columns if "title" in c.lower()][0]
conf_col = [c for c in df.columns if "conf" in c.lower()][0]
company_col = [c for c in df.columns if "company" in c.lower()]  # optional

# Compute basic stats
summary = {
    "total_rows": len(df),
    "unique_executives": df[name_col].nunique(),
    "unique_titles": df[title_col].nunique(),
}

if company_col:
    summary["unique_companies"] = df[company_col[0]].nunique()

conf_counts = df[conf_col].value_counts().to_dict()
summary["confidence_breakdown"] = conf_counts

# Display the summary neatly
print("\n=== EXECUTIVE DATASET METADATA ===")
for k, v in summary.items():
    print(f"{k:25}: {v}")

# Optionally, save to a README-like text file
with open("executive_dataset_summary.txt", "w", encoding="utf-8") as f:
    f.write("Executive Dataset Summary\n")
    f.write(f"File: executive_final.csv\n\n")
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")
    f.write("\nMain columns:\n")
    for col in [name_col, title_col, conf_col] + company_col:
        f.write(f" - {col}\n")


=== EXECUTIVE DATASET METADATA ===
total_rows               : 11680
unique_executives        : 11680
unique_titles            : 938
unique_companies         : 4656
confidence_breakdown     : {'spacy': 9243, 'high': 2236, 'low': 201}
